In [1]:
import pandas as pd



clusters = pd.read_csv("fingerprints_with_clusters.csv", usecols=["coating_id", "cluster_label"])
clusters = clusters[["coating_id", "cluster_label"]]

agg_full = pd.read_csv("agg_full_with_tsfresh.csv", parse_dates=['order_date'])
agg_full = agg_full.merge(clusters, on="coating_id", how="left")

print(agg_full['cluster_label'].isna().sum(), "Zeilen haben keine Clusterzuordnung.")


0 Zeilen haben keine Clusterzuordnung.


In [2]:
print(agg_full.head())

  order_date  coating_id  total_fill_rate  mean_fill_rate  median_fill_rate  \
0 2023-01-06          27         1.253242        0.179035          0.151515   
1 2023-01-09          27         1.273241        0.115749          0.023148   
2 2023-01-10          27         0.194583        0.048646          0.047639   
3 2023-01-11          27         2.810860        0.133850          0.116667   
4 2023-01-20          25         0.024343        0.008114          0.006667   

   min_fill_rate  max_fill_rate  std_fill_rate  total_chf    mean_chf  ...  \
0       0.080000       0.392308       0.099372     728.29  104.041429  ...   
1       0.011111       0.587500       0.179865    1345.81  122.346364  ...   
2       0.027083       0.072222       0.018482     207.31   51.827500  ...   
3       0.001852       0.555556       0.134041    3638.99  173.285238  ...   
4       0.005556       0.012121       0.003514      98.38   32.793333  ...   

   num_product_families_w10_value__median  \
0          

In [3]:
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Zielwert vorbereiten
model = agg_full.sort_values(['coating_id', 'order_date'])
model['target_total_fill_rate_next_day'] = model.groupby('coating_id')['total_fill_rate'].shift(-1)

# Features, die für den nächsten Tag vorausgeschaut werden
next_day_features = [
    "day", "weekday", "week_of_year", "month", "quarter", "year",
    "day_sin", "day_cos", "weekday_sin", "weekday_cos",
    "month_sin", "month_cos", "month_progression",
    "quarter_sin", "quarter_cos", "quarter_progression",
    "year_progression", "is_first_business_day_of_month",
    "is_last_business_day_of_month", "is_first_business_day_of_quarter",
    "is_last_business_day_of_quarter", "is_first_business_day_of_year",
    "is_last_business_day_of_year", "is_holiday", "day_before_holiday",
    "week_before_holiday", "bridgeday_flag",
    "holiday_week_including_weekends", "holiday_week_excluding_weekends"
]

for col in next_day_features:
    model[col + '_next_day'] = model.groupby('coating_id')[col].shift(-1)

# NaNs raus (letzter Tag je coating_id)
model = model.dropna(subset=['target_total_fill_rate_next_day'])

# Feature-Spalten vorbereiten
exclude_cols = ['order_date', 'target_total_fill_rate_next_day']
feature_cols = [c for c in model.columns if c not in exclude_cols and c != 'coating_id']

# Ergebnisse sammeln
cluster_results = []

for cluster in sorted(model['cluster_label'].dropna().unique()):
    print(f"\n--- Cluster {cluster} ---")

    cluster_df = model[model['cluster_label'] == cluster]
    if len(cluster_df) < 30:
        print("  → zu wenig Daten, überspringt...")
        continue

    # Zeitbasierter Split
    split_date = cluster_df['order_date'].quantile(0.8)
    train = cluster_df[cluster_df['order_date'] <= split_date]
    test  = cluster_df[cluster_df['order_date'] > split_date]

    if train.empty or test.empty:
        print("  → kein gültiger Split, überspringt...")
        continue

    X_train = train[feature_cols]
    y_train = train['target_total_fill_rate_next_day']
    X_test  = test[feature_cols]
    y_test  = test['target_total_fill_rate_next_day']

    # Kategorische Spalten one-hot encoden
    cat_cols = X_train.select_dtypes(include=['object', 'category']).columns
    X_train = pd.get_dummies(X_train, columns=cat_cols)
    X_test = pd.get_dummies(X_test, columns=cat_cols)
    X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

    # LightGBM-Dataset
    lgb_train = lgb.Dataset(X_train, y_train)
    model_params = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'learning_rate': 0.01,
        'num_leaves': 64
    }

    # Training
    model_lgbm = lgb.train(
        model_params,
        lgb_train,
        num_boost_round=500,
        valid_sets=[lgb_train]
    )

    # Vorhersage + Metriken
    y_pred = model_lgbm.predict(X_test)
    mask = ~y_test.isna() & ~pd.isna(y_pred)
    y_test_clean = y_test[mask]
    y_pred_clean = y_pred[mask]

    if len(y_test_clean) == 0:
        continue

    rmse = mean_squared_error(y_test_clean, y_pred_clean)
    mae = mean_absolute_error(y_test_clean, y_pred_clean)
    r2 = r2_score(y_test_clean, y_pred_clean)

    cluster_results.append({
        'cluster': cluster,
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'train_size': len(train),
        'test_size': len(test)
    })

# Übersicht
cluster_df_results = pd.DataFrame(cluster_results)
print("\nModellergebnisse pro Cluster:")
print(cluster_df_results.sort_values(by='r2', ascending=False))


--- Cluster 0 ---

--- Cluster 1 ---

--- Cluster 2 ---

--- Cluster 3 ---

Modellergebnisse pro Cluster:
   cluster        rmse        mae        r2  train_size  test_size
3        3    9.328698   1.435688  0.205751        7973       1989
0        0   59.043026   4.419165  0.188864        4690       1170
1        1    8.397418   1.477847  0.188598        3752        936
2        2  422.476538  15.053184 -0.153707         469        117
